# Patron de Diseno ETL / ELT en Databricks PySpark

Flujo estandar de desarrollo de procesos ETL en PySpark/Databricks.


In [ ]:
# -------------------------------------------------------------------------
# PROYECTO       : Data Warehouse Comercial
# PROCESO        : ETL_CLIENTES
# OBJETIVO       : Consolidar informacion de clientes
# VERSION        : 1.0.0
# AUTOR          : Equipo Data Engineering
# DESARROLLADOR  : Equipo Data Engineering
# FECHA CREAC.   : 2026-01-15
# FRECUENCIA     : Diaria
# CAPA           : Silver -> Gold
# -------------------------------------------------------------------------


In [ ]:
import json
import logging
from datetime import datetime

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window


In [ ]:
def get_logger(nombre):
    """Devuelve el logger estandar del proceso."""
    log = logging.getLogger(nombre)
    log.setLevel(logging.INFO)
    return log


def add_nombre_normalizado(df_origen):
    """Normaliza el nombre del cliente a mayusculas y sin espacios."""
    return df_origen.withColumn(
        "nombre_cliente",
        F.upper(F.trim(F.col("nombre_cliente")))
    )


def validate_reglas_precarga(df_origen, codigo_col, descriptivo_col):
    """Aplica las reglas de gobierno sobre los campos descriptivos.

    Marca DATO NO INFORMADO cuando falta el codigo y FUERA DE DOMINIO
    cuando el codigo existe pero no tiene descripcion asociada.
    """
    return df_origen.withColumn(
        descriptivo_col,
        F.when(
            F.col(codigo_col).isNull() | (F.trim(F.col(codigo_col)) == ""),
            F.lit("DATO NO INFORMADO")
        ).when(
            F.col(codigo_col).isNotNull()
            & (F.col(descriptivo_col).isNull()
               | (F.trim(F.col(descriptivo_col)) == "")),
            F.lit("FUERA DE DOMINIO")
        ).otherwise(F.col(descriptivo_col))
    )


def drop_duplicados_cliente(df_origen):
    """Conserva el registro mas reciente por id_cliente."""
    window_cliente = Window.partitionBy("id_cliente").orderBy(
        F.col("fecha_actualizacion").desc()
    )
    return (
        df_origen
        .withColumn("rn", F.row_number().over(window_cliente))
        .filter(F.col("rn") == 1)
        .drop("rn")
    )


def write_clientes(df_origen, destino):
    """Escribe el resultado en la tabla final en formato Delta."""
    (
        df_origen
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(destino)
    )


In [ ]:
dbutils.widgets.text("p_fecha_proceso", "")
dbutils.widgets.text("p_esquema", "prd")

var_fecha_proceso = dbutils.widgets.get("p_fecha_proceso")
var_esquema = dbutils.widgets.get("p_esquema")


In [ ]:
NOMBRE_PROCESO = "ETL_CLIENTES"
TBL_CLIENTES_SRC = f"{var_esquema}.clientes_stg"
TBL_CLIENTES_FIN = f"{var_esquema}.clientes_gold"
TBL_LOG_PROCESOS = f"{var_esquema}.log_ejecucion"

ESTADO_ACTIVO = "ACTIVO"
FORMATO_FECHA = "yyyy-MM-dd"


In [ ]:
logger = get_logger(NOMBRE_PROCESO)
logger.info("Inicio del proceso %s", NOMBRE_PROCESO)
logger.info("Parametros recibidos: fecha=%s esquema=%s",
            var_fecha_proceso, var_esquema)

df_clientes = (
    spark.table(TBL_CLIENTES_SRC)
    .select("id_cliente", "nombre_cliente", "cod_estado",
            "des_estado", "fecha_actualizacion")
    .filter(F.col("fecha_actualizacion") >= var_fecha_proceso)
)

df_transformado = (
    df_clientes
    .transform(add_nombre_normalizado)
    .transform(lambda d: validate_reglas_precarga(d, "cod_estado", "des_estado"))
    .transform(drop_duplicados_cliente)
)

write_clientes(df_transformado, TBL_CLIENTES_FIN)
logger.info("Fin del proceso %s", NOMBRE_PROCESO)
